## **NBA Prospect Evaluator**
##### *Predicting NBA success based on NCAA statistics*

In [2]:
# Test Environment
print("Jared McCain")

Jared McCain


# Data Collection

#### Use API to get college basketball player data *(getcollegedata.py)*

In [4]:
from getcollegedata import pull_all_seasons
import os
import pandas as pd

if os.path.exists("data/raw/player_stats_2015_2025.csv"):
    df = pd.read_csv(
        "data/raw/player_stats_2015_2025.csv"
    )
else:
    df = pull_all_seasons(2015, 2025)
    df.to_csv(
        "data/raw/player_stats_2015_2025.csv",
        index=False
    )

NBA Player Data

In [19]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()

all_dfs = []


if os.path.exists("data/raw/nba_player_stats_2016_2026.csv"):
    print("File already exists.")
else:
    for year in range(16, 27):  # 16 through 26
        file_path = ROOT / "data" / "raw" / f"nba_player_data_{year}.xlsx"

        print(f"Loading {file_path.name}")

        df = pd.read_excel(file_path)

        # add a season column
        df["season"] = year

        all_dfs.append(df)

    raw_player_stats = pd.concat(all_dfs, ignore_index=True)

    # Save combined dataset
    output_path = ROOT / "data" / "raw" / "nba_player_stats_2016_2026.csv"
    output_path.parent.mkdir(parents=True, exist_ok=True)

    raw_player_stats.to_csv(output_path, index=False)

    print(f"Saved to {output_path}")
    


Loading nba_player_data_16.xlsx
Loading nba_player_data_17.xlsx
Loading nba_player_data_18.xlsx
Loading nba_player_data_19.xlsx
Loading nba_player_data_20.xlsx
Loading nba_player_data_21.xlsx
Loading nba_player_data_22.xlsx
Loading nba_player_data_23.xlsx
Loading nba_player_data_24.xlsx
Loading nba_player_data_25.xlsx
Loading nba_player_data_26.xlsx
Saved to c:\Users\jftho\OneDrive\Documents\NBA_Prospect_Evaluator\data\raw\nba_player_stats_2016_2026.csv


Clean Data

In [21]:
raw_player_stats = pd.read_csv("data/raw/nba_player_stats_2016_2026.csv")
raw_player_stats.head()

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,TRB,AST,STL,BLK,TOV,PF,PTS,Trp-Dbl,Awards,season
0,1,James Harden,26,HOU,SG,82,82,3125,710,1617,...,501,612,139,51,374,229,2376,3,"MVP-9,AS",16
1,2,Stephen Curry,27,GSW,PG,79,79,2700,805,1598,...,430,527,169,15,262,161,2375,2,"MVP-1,AS,NBA1",16
2,3,Kevin Durant,27,OKC,SF,72,72,2578,698,1381,...,589,361,69,85,250,137,2029,1,"MVP-5,AS,NBA2",16
3,4,LeBron James,31,CLE,SF,76,76,2709,737,1416,...,565,514,104,49,249,143,1920,3,"MVP-3,DPOY-11,AS,NBA1",16
4,5,Damian Lillard,25,POR,PG,75,75,2676,618,1474,...,302,512,65,28,242,165,1879,0,"MVP-8,NBA2",16


In [ ]:
# Sort so the row with the most games is first
raw_player_stats = raw_player_stats.sort_values(
    ["Player", "season", "G"],
    ascending=[True, True, False]
)

# Keep only the first row for each player-season
raw_player_stats = raw_player_stats.drop_duplicates(
    subset=["Player", "season"],
    keep="first"
).reset_index(drop=True)

In [23]:
# Add columns to nba player stats df
import numpy as np

# Per-game stats
raw_player_stats["MPG"] = np.where(
    raw_player_stats["G"] > 0,
    raw_player_stats["MP"] / raw_player_stats["G"],
    np.nan
)

raw_player_stats["PPG"] = np.where(
    raw_player_stats["G"] > 0,
    raw_player_stats["PTS"] / raw_player_stats["G"],
    np.nan
)

raw_player_stats["APG"] = np.where(
    raw_player_stats["G"] > 0,
    raw_player_stats["AST"] / raw_player_stats["G"],
    np.nan
)

raw_player_stats["RPG"] = np.where(
    raw_player_stats["G"] > 0,
    raw_player_stats["TRB"] / raw_player_stats["G"],
    np.nan
)

raw_player_stats["ORPG"] = np.where(
    raw_player_stats["G"] > 0,
    raw_player_stats["ORB"] / raw_player_stats["G"],
    np.nan
)

raw_player_stats["DRPG"] = np.where(
    raw_player_stats["G"] > 0,
    raw_player_stats["DRB"] / raw_player_stats["G"],
    np.nan
)

# Per-40 stats
raw_player_stats["PTS_PER_40"] = np.where(
    raw_player_stats["MP"] > 0,
    raw_player_stats["PTS"] / raw_player_stats["MP"] * 40,
    np.nan
)

raw_player_stats["AST_PER_40"] = np.where(
    raw_player_stats["MP"] > 0,
    raw_player_stats["AST"] / raw_player_stats["MP"] * 40,
    np.nan
)

raw_player_stats["REB_PER_40"] = np.where(
    raw_player_stats["MP"] > 0,
    raw_player_stats["TRB"] / raw_player_stats["MP"] * 40,
    np.nan
)

raw_player_stats["ORB_PER_40"] = np.where(
    raw_player_stats["MP"] > 0,
    raw_player_stats["ORB"] / raw_player_stats["MP"] * 40,
    np.nan
)

raw_player_stats["DRB_PER_40"] = np.where(
    raw_player_stats["MP"] > 0,
    raw_player_stats["DRB"] / raw_player_stats["MP"] * 40,
    np.nan
)

raw_player_stats["STL_PER_40"] = np.where(
    raw_player_stats["MP"] > 0,
    raw_player_stats["STL"] / raw_player_stats["MP"] * 40,
    np.nan
)

raw_player_stats["BLK_PER_40"] = np.where(
    raw_player_stats["MP"] > 0,
    raw_player_stats["BLK"] / raw_player_stats["MP"] * 40,
    np.nan
)

raw_player_stats["TOV_PER_40"] = np.where(
    raw_player_stats["MP"] > 0,
    raw_player_stats["TOV"] / raw_player_stats["MP"] * 40,
    np.nan
)

raw_player_stats["PF_PER_40"] = np.where(
    raw_player_stats["MP"] > 0,
    raw_player_stats["PF"] / raw_player_stats["MP"] * 40,
    np.nan
)

raw_player_stats["AST_TO_TOV"] = np.where(
    raw_player_stats["TOV"] > 0,
    raw_player_stats["AST"] / raw_player_stats["TOV"],
    np.nan
)

raw_player_stats["STOCKS_PER_40"] = (
    raw_player_stats["STL_PER_40"] +
    raw_player_stats["BLK_PER_40"]
)

# Round all newly created columns to 2 decimal places
new_cols = [
    "MPG", "PPG", "APG", "RPG", "ORPG", "DRPG",
    "PTS_PER_40", "AST_PER_40", "REB_PER_40",
    "ORB_PER_40", "DRB_PER_40",
    "STL_PER_40", "BLK_PER_40",
    "TOV_PER_40", "PF_PER_40", 
    "AST_TO_TOV", "STOCKS_PER_40"
]

raw_player_stats[new_cols] = raw_player_stats[new_cols].round(2)

In [ ]:
raw_player_stats[raw_player_stats['Player'] == 'James Ennis III']

,Rk,Player,Age,Team,Pos,G,GS,MP,FG,FGA,...,AST_PER_40,REB_PER_40,ORB_PER_40,DRB_PER_40,STL_PER_40,BLK_PER_40,TOV_PER_40,PF_PER_40,AST_TO_TOV,STOCKS_PER_40
1811,349,James Ennis III,25,3TM,SF,22,5,329,54,113,...,2.55,5.11,2.55,2.55,1.95,0.61,2.31,3.40,1.11,2.55
1812,247,James Ennis III,26,MEM,SF,64,28,1501,146,321,...,1.71,6.90,1.84,5.06,1.23,0.51,1.57,4.40,1.08,1.73
1813,217,James Ennis III,27,2TM,SF,72,22,1604,182,384,...,1.77,5.59,1.82,3.77,1.17,0.45,1.40,3.34,1.27,1.62
1814,275,James Ennis III,28,2TM,SF,58,27,1230,138,294,...,1.33,5.92,1.95,3.97,1.33,0.75,1.11,4.88,1.21,2.08
1815,214,James Ennis III,29,2TM,SG,69,18,1265,160,359,...,1.99,7.75,2.25,5.50,1.11,0.70,1.83,4.33,1.09,1.80
1816,268,James Ennis III,30,ORL,SF,41,37,986,115,243,...,2.52,6.73,1.74,4.99,1.30,0.28,1.46,3.41,1.72,1.58
